# [SOLUTION] Exercise - Output structured Agent responses

In this exercise, you'll learn how to enhance your AI agent to provide structured outputs using Pydantic models. This will help ensure the agent's responses are
consistent, validated, and easily usable in downstream applications.

## Challenge

You have an existing Agent class that can:
- Process user messages
- Use tools when needed
Generate responses

Now you need to enhance it to:
- Define structured output formats using Pydantic
- Parse and validate responses
- Return data in a consistent JSON format

## Setup
First, let's import the necessary libraries:

In [ ]:

from typing import Annotated
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import json
from lib.messages import UserMessage, SystemMessage, ToolMessage  # Different message types
from lib.tooling import tool  # Tool decorator for creating AI tools
from lib.llm import LLM  # Our Language Model wrapper

from lib.parsers import ( #importing the parsers for handling different output formats
    JsonOutputParser, 
    PydanticOutputParser, 
)

In [ ]:
chat_model = LLM()


## Basic String Output
before diving into more complex output formats, let's understand how to work with simple string outputs from our Language Model.
This demonstrates the most basic form of parsing LLM responses.

In [ ]:
messages = [
    SystemMessage(content="Extract the event information."),
    UserMessage(content="Alice and Bob are going to a science fair on Friday.")
]

In [ ]:
ai_message = chat_model.invoke(messages)
ai_message

In [ ]:
parser = StrOutputParser()
print(parser.parse(ai_message))

## Working with Tools
Now let's see how we can use tools to get structured outputs from our LLM.
Tools help us enforce a specific format for the output and make it easier to process programmatically.

In [ ]:
@tool
def calendar_event(name: str, date: str, participants: list[str]):
    """Identify name of the event, date when it will happen and all the participants"""
    return {
        "name": name,
        "date": date,
        "participants": participants
    }

In [ ]:
messages = [
    SystemMessage(content="Extract the event information."),
    UserMessage(content="Alice and Bob are going to a science fair on Friday.")
]

In [ ]:
chat_model_with_tools = LLM(tools=[calendar_event])
ai_message = chat_model_with_tools.invoke(messages)


In [ ]:
# New implementation of printing the response, using parsers to handle the structured output   
parser = ToolOutputParser()
structured_output = parser.parse(ai_message)[0]["args"]
print("Structured Output:\n", structured_output)

## Using Pydantic Models for Validation
Implement Pydantic models to validate and structure the outputs from the LLM. This step ensures type safety and data integrity.

In [ ]:
class CalendarEvent(BaseModel):
    """A Pydantic model representing a calendar event."""
    # Annotated fields with descriptions and default values
    # Annotated[type, metadata]: This is a way to tell Python, 
    # "This variable is a str, but I'm also attaching some extra notes
    # to it." Python itself ignores the notes, but libraries like Pydantic 
    # or FastAPI read them to understand how to handle the data.

    name: Annotated[str, 
                    # Field(...): This is a Pydantic function. 
                    # It’s where you define the "rules" for the variable, like what 
                    # the description is or what happens if the user doesn't provide a value.
                    Field(description="Name/Title of the event. Defaults to ''", default=None)]
    date: Annotated[str, Field(description="Date of the event. Defaults to ''", default=None)]
    participants: Annotated[list[str], Field(description="Who will participate. Defaults to ''", default=None)]


In [ ]:
messages = [
    SystemMessage(content="Extract the event information."),
    UserMessage(content="Alice and Bob are going to a science fair on Friday.")
]

In [ ]:
ai_message = chat_model.invoke(input=messages, response_format=CalendarEvent)
parser = JsonOutputParser()
json_output = parser.parse(ai_message)
print("JSON Output:\n", json.dumps(json_output, indent=4))


In [ ]:
parser = PydanticOutputParser(model_class=CalendarEvent)
event: CalendarEvent = parser.parse(ai_message)
event

In [ ]:
event.participants